# AIGC 5005 · Module 3 · Part 2
## Inheritance, method overriding, and polymorphism

Companion notebook to Part 2 of the Module 3 learning content. Every example from that part is here, ready to run, with comments explaining what to notice.

**Estimated time:** 30 minutes

**How to use it:** read each explanation, run the cell beneath it, and compare the output with what you expected *before* you ran it. The prediction is where the learning happens. Change values and run cells again; you cannot break anything.

**Kernel:** select the kernel from your `module03` conda environment (top right in VS Code or Jupyter). To make one: `conda create -n module03 python=3.11 ipykernel`, then `conda activate module03`. If VS Code offers to install `ipykernel`, accept.

## 1. A superclass

`Member` is the **superclass** (also called parent or base class). Everything a library member needs goes here.

In [1]:
class Member:
    """A library member who can borrow books."""

    def __init__(self, name, borrow_limit=3):
        self.name = name
        self.borrow_limit = borrow_limit
        self.borrowed = []

    def borrow(self, title):
        if len(self.borrowed) >= self.borrow_limit:
            raise ValueError(f"{self.name} has reached the borrow limit.")
        self.borrowed.append(title)

    def describe(self):
        return f"{self.name}: {len(self.borrowed)}/{self.borrow_limit} books borrowed"

## 2. A subclass that extends it

`StaffMember(Member)` inherits every attribute and method of `Member`. It **overrides** `describe()` to add its own detail, and uses `super()` to reuse the parent's version instead of copying it.

In [2]:
class StaffMember(Member):
    """A staff member gets a higher borrow limit and a department."""

    def __init__(self, name, department, borrow_limit=10):
        super().__init__(name, borrow_limit=borrow_limit)   # reuse Member's setup, do not copy it
        self.department = department

    def describe(self):
        base = super().describe()                            # reuse the parent's answer...
        return f"{base} (staff, {self.department})"          # ...then extend it

### Polymorphism: one call, two behaviours

**Predict first:** the loop calls the same method name on two different objects. What does each line print?

In [3]:
m = Member("Aisha")
s = StaffMember("Devon", department="IT Services")

for person in (m, s):
    print(person.describe())    # the right version runs for each object's actual class

Aisha: 0/3 books borrowed
Devon: 0/10 books borrowed (staff, IT Services)


### The limits differ, the code that uses them does not

`StaffMember` inherited `borrow()` unchanged, but its `borrow_limit` is 10 instead of 3.

In [4]:
for _ in range(4):
    s.borrow("A Python book")
print(s.describe())             # Devon: 4/10 books borrowed (staff, IT Services)

try:
    for _ in range(4):
        m.borrow("A Python book")   # Member's limit is 3, so the 4th borrow raises
except ValueError as error:         # try/except only keeps the notebook running here
    print(f"Caught: {error}")

Devon: 4/10 books borrowed (staff, IT Services)
Caught: Aisha has reached the borrow limit.


## 3. Is-a checks

`isinstance(obj, Class)` is `True` when the object's class is `Class` *or a subclass of it*. `issubclass()` asks the same question about classes.

In [5]:
print(isinstance(s, Member))          # True: every StaffMember is a Member
print(isinstance(m, StaffMember))     # False: not every Member is a StaffMember
print(issubclass(StaffMember, Member))   # True

print(StaffMember.__mro__)            # the lookup order Python follows: StaffMember, then Member, then object

True
False
True
(<class '__main__.StaffMember'>, <class '__main__.Member'>, <class 'object'>)


### Watch out: forgetting `super().__init__()`

If a subclass defines `__init__` and does not call the parent's, the parent's attributes are never created.

In [6]:
class ForgetfulStaff(Member):
    def __init__(self, name, department):
        # BUG: no super().__init__(...), so self.name / borrow_limit / borrowed never exist
        self.department = department


f = ForgetfulStaff("Sam", "Library")
try:
    print(f.describe())
except AttributeError as error:       # try/except only keeps the notebook running here
    print("AttributeError:", error)   # 'ForgetfulStaff' object has no attribute 'name'

AttributeError: 'ForgetfulStaff' object has no attribute 'name'


### Try it

Add `GuestMember(Member)` whose `borrow_limit` is fixed at `1` no matter what is passed in, and whose `describe()` adds `"(guest)"` at the end. Create one instance of each of the three classes and print `person.describe()` for all three in a loop. Write it in the cell below.

In [7]:
# Your GuestMember here, then the loop over Member, StaffMember and GuestMember objects.

### Is-a or has-a?

Inheritance models **is-a** (a staff member *is a* member). When one thing merely **has** another, store it as an attribute instead.

In [8]:
class Library:
    """A Library HAS books; it is not a kind of book, so it holds them in an attribute."""

    def __init__(self, name):
        self.name = name
        self.books = []             # has-a: a list of Book-like items

    def add(self, title):
        self.books.append(title)


lib = Library("Main Branch")
lib.add("Fluent Python")
print(lib.name, lib.books)

Main Branch ['Fluent Python']
